# German Credit Bureau Default Risk Scoring

**Objective:** Clean the real German Credit dataset, engineer risk features, and build a scoring model to flag high-risk borrowers.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

### 1. Load Real Dataset (UCI German Credit)
We use the openml API to fetch the German Credit Risk dataset.

In [2]:
# Fetch data
credit_data = fetch_openml('credit-g', version=1, as_frame=True)
df = credit_data.frame
print(f"Dataset shape: {df.shape}")
df.head()

### 2. Preprocessing & Feature Engineering

In [3]:
# Target encoding (good = 0, bad = 1)
df['default'] = df['class'].apply(lambda x: 1 if x == 'bad' else 0)
df = df.drop('class', axis=1)

# One-hot encoding categorical variables
cat_cols = df.select_dtypes(include=['category', 'object']).columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.head()

### 3. Model Training

In [4]:
X = df.drop('default', axis=1)
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.4).astype(int) 

In [5]:
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.3f}\n")
print(classification_report(y_test, y_pred))

In [6]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.title('Confusion Matrix')
plt.show()